# 06 — Zero-Shot Desire Categories

The project's **headline original contribution**: labels reviews along the candidate desire dimensions without any supervised training, using NLI-based zero-shot classification.

Default model is `MoritzLaurer/deberta-v3-base-zeroshot-v2.0` (smaller / faster than `-large-`); swap to the large variant if a GPU with ≥16GB is available. Inference is slow on CPU — keep the sample size small (default 2,000).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from utils import ARTIFACTS_DIR, FIGURES_DIR, save_metrics, set_seed
set_seed()
sns.set_theme(style="whitegrid")

ZSL_MODEL = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
N_SAMPLE  = 2_000

CANDIDATES = [
    "status seeking",
    "fear of missing out",
    "genuine need",
    "social pressure",
    "hedonic impulse",
    "advertising language echo",
]

In [ ]:
df = pl.read_parquet(ARTIFACTS_DIR / "split_test.parquet").to_pandas()
df = df.sample(N_SAMPLE, random_state=42).reset_index(drop=True)
print(df.shape)

In [ ]:
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1

zsl = pipeline(
    "zero-shot-classification",
    model=ZSL_MODEL,
    device=device,
)
print("Loaded", ZSL_MODEL, "on device", device)

In [ ]:
# Multi-label (allow co-occurring categories)
out = zsl(
    df["review"].tolist(),
    candidate_labels=CANDIDATES,
    multi_label=True,
    batch_size=8,
)

scores = []
for o in out:
    row = {label: score for label, score in zip(o["labels"], o["scores"])}
    scores.append([row[c] for c in CANDIDATES])

scores_df = pd.DataFrame(scores, columns=[c.replace(" ", "_") for c in CANDIDATES])
desire = pd.concat([df[["label", "review"]].reset_index(drop=True), scores_df], axis=1)
desire.head(3)

In [ ]:
# Cross-tabulation: mean desire-category probability by polarity class
class_means = desire.groupby("label")[scores_df.columns.tolist()].mean()
print(class_means)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(class_means, annot=True, fmt=".3f", cmap="vlag", center=0.5, ax=ax,
            yticklabels=["negative (0)", "positive (1)"])
ax.set_title("Mean zero-shot desire-category probability by polarity")
plt.tight_layout()
fig_path = FIGURES_DIR / "06_zeroshot_desire_by_polarity.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Top-k examples per category (qualitative, useful for Appendix)
top_per_cat = {}
for c in scores_df.columns:
    idx = desire[c].nlargest(3).index
    top_per_cat[c] = desire.loc[idx, ["label", c, "review"]].to_dict("records")

import json
(ARTIFACTS_DIR / "zeroshot_examples.json").write_text(json.dumps(top_per_cat, indent=2))
print("Wrote zeroshot_examples.json with top-3 per category.")

In [ ]:
desire.to_parquet(ARTIFACTS_DIR / "zeroshot_desire.parquet")
save_metrics("zeroshot_desire", {
    "model": ZSL_MODEL,
    "n_sample": N_SAMPLE,
    "candidates": CANDIDATES,
    "class_means": class_means.to_dict(),
})

## Findings to record

- The class-conditional heatmap (`figures/06_zeroshot_desire_by_polarity.png`) is the project's signature figure: it shows which desire categories load on positive vs. negative reviews.
- Per-category top-3 examples (`zeroshot_examples.json`) are strong Appendix material — pick a few to embed in the Discussion as qualitative evidence.
- Validate against a hand-labeled gold set of ~50 reviews per category to report inter-rater agreement; this is the cleanest path to a human-validation paragraph in the Discussion.